# Phase 6 dry run: the GPU systems on the pilot answers

Runs Channel A and the trained repairer on the 120 pilot answers (development issuers) and stores every output: Channel A's spans and fusion features, and the repair of each answer with the rule checker's spans and with Channel A's. Attach the pilot items dataset (no gold), the faithguard-detector-models dataset and the stage A notebook's output. The outputs are scored on a laptop against the gold store. Expect under 1 GPU hour.

**On Kaggle:** Create → New Notebook → File → Import Notebook → this file. Settings: *Accelerator* **GPU T4 x2**, *Internet* **on**.
For long jobs use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours and keeps everything in `/kaggle/working/outputs`.
To continue in a later notebook, add this notebook's output as input (*Add Input → Your Work*).

**Afterwards:** download `outputs/*/results.json` and `report.md` files into `runs/` in the repository, and log the session's GPU hours in `logs/gpu-hours.csv`. Checkpoints are saved regularly; re-running the same cell resumes from the latest one.

In [ ]:
# Setup: clone FaithGuard, install it with the pinned GPU packages, fetch the data.
import shutil, socket
try:
    socket.gethostbyname("pypi.org")
except OSError:
    raise RuntimeError("No internet in this session. Kaggle: verify your phone number (Settings), then switch Internet on in the right-hand panel.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU in this session. Kaggle: verify your phone number (Settings), then set Accelerator to GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
if not os.path.exists('faithguard'):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/SharafThawfeek/FaithGurd-new.git', 'faithguard'])
os.chdir('faithguard')
REQUIREMENTS = ['transformers==5.19.0', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'huggingface_hub==1.33.0', 'datasets==5.1.0', 'lettucedetect==0.2.3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
# Kaggle's image ships torchao 0.10, which PEFT 0.21 rejects when it adds LoRA to fp16 weights; nothing here uses it
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
# this kernel started before the install, and the clone's own folder would shadow the package: import from src/
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
OUT = os.path.join(WORK, 'outputs'); os.makedirs(OUT, exist_ok=True)
!faithguard data download all
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Inputs: the pilot items, Channel A's model and stage A's adapter

In [ ]:
import glob, os
for flat in glob.glob('/kaggle/input/**/*__model__*', recursive=True):  # the models dataset, folders flattened
    target = os.path.join(WORK, 'prev', *os.path.basename(flat).split('__'))
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        os.symlink(flat, target)
CHANNEL_A = os.path.join(WORK, 'prev', 'channel-a', 'model')
ADAPTER = sorted(glob.glob('/kaggle/input/**/repair-sft/adapter', recursive=True))[0]
ITEMS = sorted(glob.glob('/kaggle/input/**/items.jsonl', recursive=True))[0]
print(ITEMS, CHANNEL_A, os.path.exists(CHANNEL_A), ADAPTER)

## Run the systems on every item

In [ ]:
!python -m faithguard.train.systems --items $ITEMS --channel-a $CHANNEL_A --adapter $ADAPTER --out {OUT}/systems-pilot

## Results

In [ ]:
import glob, json
for path in sorted(glob.glob(OUT + '/**/results.json', recursive=True)):
    print(path)
    print(json.dumps({k: v for k, v in json.load(open(path)).items() if k in ('summary', 'channel_a', 'channel_b', 'auroc', 'false_alarm_clean', 'wrong_context_recall')}, indent=1)[:1500])